# Hybrid RAG: The AI Recruiter (Structured + Semantic)

**Copyright 2026, Denis Rothman**

This notebook implements the **Retrieval-Augmented Generation (RAG)** phase for our Hybrid HR System.

Unlike the previous chapter's "Cyber Incident" agent—which searched for pure text—this "AI Recruiter" must respect **Structured Constraints**. It is not enough to find a candidate who *sounds* like a leader; they must also fit within the **Salary Budget** and meet the **Experience Requirements**.

### Notebook Flow:

* **1. The Hybrid Query:** We accept a multi-modal input from the user (e.g., *"Find me a Python expert"* + *Max Salary $150k*).
* **2. Context Injection:** The system dynamically retrieves the correct "Hiring Persona" (e.g., *Technical Screener* vs. *Culture Fit Officer*) from the `RECRUITMENT_RULES` table.
* **3. Hybrid Retrieval:** We execute a specialized Oracle SQL query that combines a strict `WHERE` clause (filtering by scalar data) with a `VECTOR_DISTANCE` sort (ranking by semantic similarity).
* **4. Augmented Generation:** The LLM reviews the retrieved candidate profiles and generates a hiring recommendation based on the active persona.

# 1.Installation and Setup

## Installing OpenAI

Move this 3.1 cell here to avoid dependencies conflicts during the instllation if you encounter any.

In [ ]:
#!pip install tqdm==4.67.1 --upgrade
!pip install openai==2.14.0

## Setting Oracle up

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
wallet_path = "/content/drive/MyDrive/oracle_wallet"

In [ ]:
# 1.Installation and Setup
# -------------------------------------------------------------------------
# We install specific versions for stability and reproducibility.
# We include tiktoken for token-based chunking and tenacity for robust API calls.

In [ ]:
!pip install oracledb==3.4.1

In [ ]:
import oracledb
print(oracledb.__version__)

## Oracle connection

In [ ]:
import os
from google.colab import userdata
oracle_dsn = userdata.get('oracle_dsn')

creds_path = "/content/drive/MyDrive/files/oracle/credentials.txt"
if not os.path.exists(creds_path):
    raise FileNotFoundError(f"Credentials file not found: {creds_path}")

def read_key_value_file(path):
    creds = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            k, v = line.split("=", 1)
            creds[k.strip()] = v.strip()
    return creds

creds = read_key_value_file(creds_path)

# Use values (do not print them)
user = creds.get("user")
password = creds.get("password")
wallet_password = creds.get("wallet_password")
dsn = creds.get("dsn", oracle_dsn)
wallet_path = creds.get("wallet_path", "/content/drive/MyDrive/files/oracle")

import oracledb
connection = oracledb.connect(
    user=user,
    password=password,
    dsn=dsn,
    config_dir=wallet_path,
    wallet_location=wallet_path,
    wallet_password=wallet_password
)

cursor = connection.cursor()
cursor.execute("SELECT 'Connected!' FROM dual")
print(cursor.fetchone())

##  Checking the Oracle tables

In [ ]:
# Verify that the tables exist and the VECTOR columns are ready
print("Checking Oracle HR tables...\n")

cursor.execute("""
SELECT table_name
FROM user_tables
WHERE table_name IN ('CANDIDATE_POOL', 'RECRUITMENT_RULES')
""")

tables = cursor.fetchall()
print("Tables found:", tables)

print("\n--- Column Check ---")
for table in ['CANDIDATE_POOL', 'RECRUITMENT_RULES']:
    print(f"\nChecking {table}:")
    try:
        cursor.execute(f"SELECT column_name, data_type FROM user_tab_columns WHERE table_name = '{table}'")
        for row in cursor.fetchall():
            print(f"  - {row[0]}: {row[1]}")
    except Exception as e:
        print(f"  Error: {e}")

In [ ]:
# Imports and API Key Setup
# We will use the OpenAI library to interact with the LLM and Google Colab's
# secret manager to securely access your API key.

import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    api_key = userdata.get("API_KEY")
    if not api_key:
        raise userdata.SecretNotFoundError("API_KEY not found.")

    # Set environment variable for downstream tools/libraries
    os.environ["OPENAI_API_KEY"] = api_key

    # Create client (will read from OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI API key loaded and environment variable set successfully.")

except userdata.SecretNotFoundError:
    print('Secret "API_KEY" not found.')
    print('Please add your OpenAI API key to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the API key: {e}")

# Configuration
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536 # Dimension for text-embedding-3-small
GENERATION_MODEL = "gpt-5.2"

## Miscellaneous

In [ ]:
# Imports for this notebook
import json
import time
from tqdm.auto import tqdm
import tiktoken
from tenacity import retry, stop_after_attempt, wait_random_exponential
# general imports required in the notebooks of this book
import re
import textwrap
from IPython.display import display, Markdown
import copy

# 2.Helper Functions for Chunking and Embedding

In [ ]:
# Helper Functions for Chunking and Embedding
# -------------------------------------------------------------------------

# Initialize tokenizer for robust, token-aware chunking
tokenizer = tiktoken.get_encoding("cl100k_base")

def chunk_text(text, chunk_size=400, overlap=50):
    """Chunks text based on token count with overlap (Best practice for RAG)."""
    tokens = tokenizer.encode(text)
    chunks = []
    for i in range(0, len(tokens), chunk_size - overlap):
        chunk_tokens = tokens[i:i + chunk_size]
        chunk_text = tokenizer.decode(chunk_tokens)
        # Basic cleanup
        chunk_text = chunk_text.replace("\n", " ").strip()
        if chunk_text:
            chunks.append(chunk_text)
    return chunks

@retry(wait=wait_random_exponential(min=1, max=60), stop=stop_after_attempt(6))
def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    """Generates embeddings for a batch of texts using OpenAI, with retries."""
    # OpenAI expects the input texts to have newlines replaced by spaces
    texts = [t.replace("\n", " ") for t in texts]
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]


# 3.Dynamic Context Retrieval: The Hiring Persona

In a production RAG system, "Context" isn't just what you find in a document; it's also **who the AI is supposed to be**.

Instead of hardcoding prompts like *"You are a helpful assistant"*, we fetch specific **Hiring Personas** from our `RECRUITMENT_RULES` table. This allows the organization to centrally manage how different recruiters (Technical vs. Cultural) behave.

The code below queries the database to see which personas are available for our session.

In [ ]:
# Fetch available personas from the database
print("--- Retrieving Active Hiring Personas from Oracle DB ---\n")

cursor.execute("SELECT rule_id, agent_persona, evaluation_criteria FROM recruitment_rules")
rules = cursor.fetchall()

# We store them in a dictionary for easy retrieval later
active_personas = {}

for r in rules:
    r_id, persona_lob, criteria_lob = r

    # FIX: Check if it's a LOB and read it, otherwise use as string
    persona = persona_lob.read() if hasattr(persona_lob, 'read') else str(persona_lob)
    criteria = criteria_lob.read() if hasattr(criteria_lob, 'read') else str(criteria_lob)

    active_personas[r_id] = {
        "persona": persona,
        "criteria": criteria
    }

    print(f"🆔 ID: {r_id}")
    print(f"👤 Persona: {persona}")
    print(f"📋 Criteria Snippet: {criteria[:80]}...")
    print("-" * 50)

print(f"\n✅ Loaded {len(active_personas)} personas into application memory.")

# 4.The Hybrid Search Engine

We now define the retrieval function `find_candidates_hybrid`. This is the engine of our application.

It performs two operations simultaneously in the Oracle Database:
1.  **Vector Search:** Compares the meaning of your query (e.g., "Python expert") against the `resume_vector`.
2.  **SQL Filtering:** Applies hard constraints on `salary_expectation` and `years_experience`.

This ensures we don't just find "good matches," but "feasible matches."

In [ ]:
def find_candidates_hybrid(user_query, max_salary=1000000, min_experience=0):
    """
    Performs a Hybrid Search:
    1. VECTOR: Semantically matches the user_query against candidate resumes.
    2. SQL: Filters out candidates who don't meet the salary/experience constraints.
    """
    print(f"   Searching for: '{user_query}'")
    print(f"   Constraints: Salary <= ${max_salary:,} | Exp >= {min_experience} years")

    # 1. Vectorize the query using our helper function
    query_vector = get_embeddings_batch([user_query])[0]

    # 2. Prepare the Hybrid SQL
    # We select the candidate details AND the distance score
    # We use DOT product for similarity (Higher is better)
    sql = """
        SELECT candidate_id, full_name, years_experience, salary_expectation, summary,
               VECTOR_DISTANCE(resume_vector, :v, DOT) as similarity
        FROM candidate_pool
        WHERE salary_expectation <= :max_sal
          AND years_experience >= :min_exp
        ORDER BY similarity DESC
        FETCH FIRST 3 ROWS ONLY
    """

    # 3. Execute
    # Crucial: Tell Oracle the :v bind variable is a vector
    cursor.setinputsizes(v=oracledb.DB_TYPE_VECTOR)

    cursor.execute(sql, {
        "v": query_vector,
        "max_sal": max_salary,
        "min_exp": min_experience
    })

    results = cursor.fetchall()
    print(f"   -> Found {len(results)} candidates fitting criteria.\n")
    return results

In [ ]:
# Unit Test: Verify the Hybrid Engine works
# We look for a "Python Developer" who is affordable (< $150k).

test_query = "Experienced Python Developer"
test_salary = 150000
test_exp = 2

print(f"🧪 Testing Engine with: '{test_query}' (Max Salary: ${test_salary:,})")

results = find_candidates_hybrid(test_query, max_salary=test_salary, min_experience=test_exp)

print("--- Test Results ---")
for r in results:
    c_id, name, exp, sal, summary_lob, score = r

    # Handle LOB conversion safely
    summary_text = summary_lob.read() if hasattr(summary_lob, 'read') else str(summary_lob)

    print(f"✅ Found: {name}")
    print(f"   Salary: ${sal:,}")
    print(f"   Match Score: {score:.4f}")
    print("-" * 30)

In [ ]:
print("\n=== Oracle HR System Status ===\n")

# 1. Verify Candidate Pool
print("--- Table: CANDIDATE_POOL ---")
# FIX: Count rows where vector is not null, rather than counting the vector column itself
cursor.execute("""
    SELECT COUNT(*),
           SUM(CASE WHEN resume_vector IS NOT NULL THEN 1 ELSE 0 END)
    FROM candidate_pool
""")
total_cand, total_vec_cand = cursor.fetchone()
print(f"Total Candidates: {total_cand}")
print(f"Vectorized:       {total_vec_cand}")

# Sample Data
cursor.execute("SELECT full_name, salary_expectation, years_experience FROM candidate_pool FETCH FIRST 2 ROWS ONLY")
for row in cursor.fetchall():
    print(f"   - {row[0]}: ${row[1]:,} | {row[2]} yrs exp")


# 2. Verify Recruitment Rules
print("\n--- Table: RECRUITMENT_RULES ---")
# FIX: Same count logic for rules
cursor.execute("""
    SELECT COUNT(*),
           SUM(CASE WHEN rule_vector IS NOT NULL THEN 1 ELSE 0 END)
    FROM recruitment_rules
""")
total_rules, total_vec_rules = cursor.fetchone()
print(f"Total Personas: {total_rules}")
print(f"Vectorized:     {total_vec_rules}")

# Sample Data
cursor.execute("SELECT rule_id, agent_persona FROM recruitment_rules FETCH FIRST 2 ROWS ONLY")
for row in cursor.fetchall():
    rule_id, persona_lob = row
    # Handle LOB safely
    persona = persona_lob.read() if hasattr(persona_lob, 'read') else str(persona_lob)
    print(f"   - {rule_id}: {persona[:60]}...")

print("\n=== System Ready ===")

# 5.The AI Recruiter (Augmented Generation)

This is the final assembly of our RAG pipeline. We define the function `generate_hiring_recommendation`.

It connects all three components we have built:
1.  **The Context:** It looks up the specific **Persona** (e.g., "Culture Fit Officer") you selected.
2.  **The Retrieval:** It calls our **Hybrid Engine** to find candidates who match your query AND budget.
3.  **The Generation:** It constructs a dynamic prompt, forcing the LLM to evaluate the retrieved candidates *strictly* according to the persona's rules.

In [ ]:
def generate_hiring_recommendation(user_query, max_salary, min_exp, persona_id="rule_tech_screener"):

    # 1. Validation
    if persona_id not in active_personas:
        return f"❌ Error: Persona '{persona_id}' not found. Available: {list(active_personas.keys())}"

    # 2. Retrieve Context (Who is the AI?)
    persona_data = active_personas[persona_id]
    system_persona = persona_data['persona']
    grading_rubric = persona_data['criteria']

    print(f"🤖 ACTIVATING AGENT: {persona_id}")
    print(f"   Goal: {system_persona}")

    # 3. Retrieve Evidence (The Hybrid Search)
    candidates = find_candidates_hybrid(user_query, max_salary, min_exp)

    if not candidates:
        return "⚠️ No candidates found matching these strict criteria."

    # 4. Augment the Prompt (Context Injection)
    # We turn the database rows into a readable text block for the LLM
    context_block = ""
    for c in candidates:
        c_id, name, exp, sal, summary_lob, score = c
        # Safe LOB read
        summary = summary_lob.read() if hasattr(summary_lob, 'read') else str(summary_lob)

        context_block += f"""
        --- CANDIDATE PROFILE ---
        ID: {c_id}
        Name: {name}
        Cost: ${sal:,} (Budget: ${max_salary:,})
        Experience: {exp} years
        Resume Summary: {summary}
        (Vector Match Score: {score:.4f})
        -------------------------
        """

    # 5. Construct the Final Prompt
    user_message = f"""
    USER REQUEST: "{user_query}"

    CANDIDATES FOUND (Database Output):
    {context_block}

    INSTRUCTIONS:
    Based on your persona rules below, evaluate these candidates.
    1. Select the BEST fit.
    2. Explain WHY, referencing their specific skills.
    3. If they are over budget or underqualified, mention it as a risk.

    YOUR GRADING RUBRIC:
    {grading_rubric}
    """

    # 6. Generate (Call OpenAI)
    print("🧠 analyzing candidates via GPT...")

    response = client.chat.completions.create(
        model="gpt-5.2",
        messages=[
            {"role": "system", "content": system_persona},
            {"role": "user", "content": user_message}
        ],
        temperature=0.3 # Low temperature for factual evaluation
    )

    return response.choices[0].message.content

# 6.Live Simulation: The AI Recruiter in Action

In this final section, we run the complete pipeline.

We have defined three control variables below:
1.  `search_query`: What are we looking for? (Vector Search)
2.  `max_budget`: What is the hard limit? (SQL Filter)
3.  `agent_role`: Which AI persona should evaluate the results? (Context Injection)

**Experiment:** Try changing the `agent_role` from `rule_tech_screener` (who cares about skills) to `rule_culture_screener` (who cares about soft skills) and see how the AI's recommendation changes for the *exact same candidates*.

In [ ]:
# @title 🎯 Run the Live Recruitment Simulation
# -------------------------------------------------------------------------

# 1. Define your Search Parameters
search_query = "We need a Python Backend developer who can lead a team." # @param {type:"string"}
max_budget = 160000 # @param {type:"slider", min:80000, max:250000, step:5000}
min_experience = 4 # @param {type:"slider", min:0, max:20, step:1}

# 2. Select the AI Persona (Dropdown)
# Options: 'rule_tech_screener', 'rule_culture_screener', 'rule_junior_scout'
agent_role = "rule_tech_screener" # @param ["rule_tech_screener", "rule_culture_screener", "rule_junior_scout"]

print(f"🎬 STARTING SIMULATION")
print(f"   Query: '{search_query}'")
print(f"   Budget: ${max_budget:,}")
print(f"   Agent: {agent_role}\n")
print("=" * 60)

# 3. Execute the Full Pipeline
recommendation = generate_hiring_recommendation(
    user_query=search_query,
    max_salary=max_budget,
    min_exp=min_experience,
    persona_id=agent_role
)

# 4. Display the AI's Verdict
print("\n" + "=" * 60)
print(f"📢 AI RECOMMENDATION ({agent_role})")
print("=" * 60)
print(recommendation)